In [1]:
from point_cloud_classifier.classifier import PointCloudClassifier, DataClassifierFormat
import os

dir = r"D:\Raphael\Essais\2_TUILES_DE_REFERENCE"
point_cloud_paths = [os.path.join(dir, file) for file in os.listdir(dir)[0::3] if file.endswith('.laz')]
_ , X, Y = DataClassifierFormat.load_data(point_cloud_paths, return_classification=True, fraction_of_dataset=3e-3, is_random=False)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Loading dataset: 100%|██████████| 7/7 [22:11<00:00, 190.18s/pointcloud]


In [2]:
import numpy as np

SITN_REMAP = {
    "default": 0, 
    2:  2, 
    18: 2, 
    31: 2, 
    3:  3, 
    4:  3, 
    5:  3, 
    6:  6, 
    21: 21, 
    22: 22, 
    26: 26
}
def remap(labels: np.ndarray, mapping: dict) -> np.ndarray:
    default = mapping.get("default", None)

    if default == "keep":
        return np.vectorize(lambda x: mapping.get(x, x))(labels)
    elif default is not None:
        return np.vectorize(lambda x: mapping.get(x, default))(labels)
    else:
        return np.vectorize(lambda x: mapping.get(x, 0))(labels)

Y_corrected = remap(Y, SITN_REMAP)

output_dir = r"D:\Raphael\point-cloud-classifier\data\tabulated_data"

os.makedirs(output_dir, exist_ok=True)

features_path = os.path.join(output_dir, "X.npy")
labels_path = os.path.join(output_dir, "y.npy")

# 4. Save the arrays
np.save(features_path, X)
np.save(labels_path, Y_corrected)

print(f"Arrays successfully saved to {output_dir}")


Arrays successfully saved to D:\Raphael\point-cloud-classifier\data\tabulated_data


In [4]:
from sklearn.ensemble import RandomForestClassifier
import joblib

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y_corrected, test_size=0.2, random_state=42
)


# 3. Entraînement avec gestion du déséquilibre
# 'balanced' calcule automatiquement les poids pour vos classes 2, 3, 6 et 21
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# save model
model_dir = r"D:\Raphael\point-cloud-classifier\data\tabulated_data\models"
os.makedirs(model_dir, exist_ok=True)

# 3. Définir le chemin complet du fichier
model_path = os.path.join(model_dir, "RF_separated.joblib")

# 4. Sauvegarder le modèle sur le disque
joblib.dump(model, model_path)

print(f"Modèle entraîné et sauvegardé avec succès dans : {model_path}")

Modèle entraîné et sauvegardé avec succès dans : D:\Raphael\point-cloud-classifier\data\tabulated_data\models\RF_separated.joblib


In [5]:
import json
import os
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
)

# Votre dictionnaire de correspondance complet
PROJECT_CLASSIFIED_MAP = {
    0:  "Other",
    2:	"Ground",
    3:	"vegetation",
    6:	"Building roofs",
    21:	"Cars",
    22:	"Building facades",
    26:	"Roof structures",
}

# 1. Générer les prédictions
y_pred = model.predict(X_test)

# 2. Extraire et trier uniquement les étiquettes présentes dans vos données de test/modèle
# scikit-learn trie toujours les classes par ordre numérique croissant (2, 3, 6, 21)
active_labels = sorted(list(model.classes_))
class_names = [PROJECT_CLASSIFIED_MAP[label] for label in active_labels]

# 3. Calculer le rapport scikit-learn sous forme de dictionnaire Python
# target_names applique automatiquement "Ground", "vegetation", etc. dans le bon ordre
report_dict = classification_report(
    y_test, y_pred, labels=active_labels, target_names=class_names, output_dict=True
)

# 4. Construire la structure finale attendue
metrics_report = {
    "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
    "cohen_kappa": float(cohen_kappa_score(y_test, y_pred)),
    "report_dict": report_dict,
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=active_labels).tolist(),
}

# 5. Sauvegarder au format JSON sur votre disque D:\
output_dir = r"D:\Raphael\point-cloud-classifier\data\tabulated_data\models"
os.makedirs(output_dir, exist_ok=True)
json_path = os.path.join(output_dir, "metrics_report_separated.json")

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_report, f, indent=4, ensure_ascii=False)

print(f"Rapport JSON avec les noms de classes sauvegardé dans : {json_path}")


Rapport JSON avec les noms de classes sauvegardé dans : D:\Raphael\point-cloud-classifier\data\tabulated_data\models\metrics_report_separated.json


## Predict single point_cloud

In [4]:
from tqdm import tqdm
import os
import numpy as np
import json
from point_cloud_classifier.classifier import DataClassifierFormat
from point_cloud_classifier.helper import visualize_point_cloud_classification, remap
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
)
import gc
import joblib

model = joblib.load(r"D:\Raphael\point-cloud-classifier\data\tabulated_data\models\RF_separated.joblib")

PROJECT_CLASSIFIED_MAP = {
    0:  "Other",
    2:	"Ground",
    3:	"vegetation",
    6:	"Building roofs",
    21:	"Cars",
    22:	"Building facades",
    26:	"Roof structures",
}

SITN_REMAP = {
    "default": 0, 
    2:  2, 
    18: 2, 
    31: 2, 
    3:  3, 
    4:  3, 
    5:  3, 
    6:  6, 
    21: 21, 
    22: 22, 
    26: 26
}

dir = r"D:\Raphael\Essais\2_TUILES_DE_REFERENCE"

output_pc_dir = r"D:/Raphael/point-cloud-classifier/data/point_clouds"

already_done_list = os.listdir(output_pc_dir)

files_to_process = [f for i, f in enumerate(os.listdir(dir)) if i % 3 != 0]

for file in tqdm(files_to_process, unit = "file", desc="Infere Multi-Class Classifier"):
    if not file.endswith(('.laz', '.las')) or (file in already_done_list):
        continue

    try:
        pc_path = os.path.join(dir, file)
        points , X, Y = DataClassifierFormat.load_data(pc_path, return_classification=True, fraction_of_dataset=1, is_random=False)
        Y = remap(Y, SITN_REMAP)
        y_pred = model.predict(X)

        visualize_point_cloud_classification(points, y_pred, f"D:/Raphael/point-cloud-classifier/data/point_clouds/{file.split('.')[0]}.copc.laz")

        active_labels = sorted(list(model.classes_))
        class_names = [PROJECT_CLASSIFIED_MAP[label] for label in active_labels]

        report_dict = classification_report(
            Y, y_pred, labels=active_labels, target_names=class_names, output_dict=True
        )

        # 4. Construire la structure finale attendue
        metrics_report = {
            "balanced_accuracy": float(balanced_accuracy_score(Y, y_pred)),
            "cohen_kappa": float(cohen_kappa_score(Y, y_pred)),
            "report_dict": report_dict,
            "confusion_matrix": confusion_matrix(Y, y_pred, labels=active_labels).tolist(),
        }

        # 5. Sauvegarder au format JSON sur votre disque D:\
        output_dir = r"D:\Raphael\point-cloud-classifier\data\tabulated_data\models"
        os.makedirs(output_dir, exist_ok=True)
        json_path = os.path.join(output_dir, f"{file.split('.')[0]}_multi-classRF.json")

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(metrics_report, f, indent=4, ensure_ascii=False)

        print(f"Rapport JSON avec les noms de classes sauvegardé dans : {json_path}")

    except MemoryError:
        # Intercepte l'erreur d'allocation RAM, nettoie immédiatement ce qui traîne et passe au fichier suivant
        print(f"\n[ATTENTION] Mémoire insuffisante pour traiter le fichier : {file}. Passage au suivant.")
        
    finally:
        # S'exécute TOUJOURS (que l'itération ait réussi ou échoué) pour garantir la libération de la RAM
        # On utilise 'locals()' pour éviter une erreur au cas où le MemoryError est survenu avant la création d'une variable
        for var_name in ['points', 'X', 'Y', 'y_pred', 'metrics_report', 'report_dict']:
            if var_name in locals():
                del locals()[var_name]
        gc.collect()


Loading dataset: 100%|██████████| 1/1 [20:57<00:00, 1257.57s/pointcloud]
d:\Raphael\point-cloud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Raphael\point-cloud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Raphael\point-cloud-classifier\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(av

Rapport JSON avec les noms de classes sauvegardé dans : D:\Raphael\point-cloud-classifier\data\tabulated_data\models\2529000_1198500_multi-classRF.json


Loading dataset: 100%|██████████| 1/1 [03:27<00:00, 207.96s/pointcloud]16.72s/file]


Rapport JSON avec les noms de classes sauvegardé dans : D:\Raphael\point-cloud-classifier\data\tabulated_data\models\2534500_1194500_multi-classRF.json


Loading dataset: 100%|██████████| 1/1 [04:23<00:00, 263.46s/pointcloud]75.93s/file]


...

Loading dataset: 100%|██████████| 1/1 [03:06<00:00, 186.28s/pointcloud]31.32s/file]


Rapport JSON avec les noms de classes sauvegardé dans : D:\Raphael\point-cloud-classifier\data\tabulated_data\models\2567000_1206500_multi-classRF.json


Loading dataset: 100%|██████████| 1/1 [02:33<00:00, 153.96s/pointcloud]09.66s/file]


Rapport JSON avec les noms de classes sauvegardé dans : D:\Raphael\point-cloud-classifier\data\tabulated_data\models\2569000_1210000_multi-classRF.json


Infere Multi-Class Classifier: 100%|██████████| 12/12 [2:49:42<00:00, 848.56s/file]
